# Ice Velocity

* Description: This dataset contains monthly gridded ice velocity maps of the Antarctic Ice Sheet derived from Sentinel-1 data acquired between 2014-01-01 and 2021-12-31. 
* Original Data Source: https://opensciencedata.esa.int/products/ice-sheet-velocity-antarctic-2021/collection
* Reference: https://climate.esa.int/en/projects/ice-sheets-antarctic/
* OSC entry: https://opensciencedata.esa.int/products/ice-sheet-velocity-antarctic-2021/collection
* License: CC-BY-NC-4.0
* Repo Folder: ./datasets/ice_velocity

In [25]:
import numpy as np
import cartopy.crs as ccrs
import xarray as xr
import hvplot.xarray

url = 'https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/polar_cube_datasets/ice_velocity/ice_velocity.zarr/'

In [26]:
ds = xr.open_zarr(url, chunks={})
ds

<xarray.Dataset> Size: 2TB
Dimensions:                              (time: 87, y: 24580, x: 28680)
Coordinates:
  * time                                 (time) datetime64[ns] 696B 2014-10-0...
  * y                                    (y) float64 197kB 2.458e+06 ... -2.4...
  * x                                    (x) float64 229kB -2.868e+06 ... 2.8...
    crs                                  int32 4B ...
Data variables:
    land_ice_surface_easting_stddev      (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_easting_velocity    (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_measurement_count   (time, y, x) float64 491GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_northing_stddev     (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_northing_velocity   (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_velocity_magnitude  (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
    land_ice_surface_vertical_velocity   (time, y, x) float32 245GB dask.array<chunksize=(5, 1892, 2208), meta=np.ndarray>
Attributes: (12/37)
    title:                     Ice Velocity of the Antarctic Ice Sheet
    institution:               ENVEO IT GmbH
    source:                    Copernicus Sentinel-1A and Sentinel-1B
    history:                   Initial product version 1.3
    references:                https://climate.esa.int/en/projects/ice-sheets...
    tracking_id:               7820956a-a125-4b94-a05f-f5161435aa24
    ...                        ...
    standard_name_vocabulary:  CF Standard Name Table v77
    license:                   ESA CCI Data Policy: free and open access
    platform:                  SENTINEL_1A, SENTINEL_1B
    sensor:                    SAR
    spatial_resolution:        200m
    key_variables:             Ice Velocity

In [31]:
vx = "land_ice_surface_easting_velocity"
vy = "land_ice_surface_northing_velocity"
proj3031 = ccrs.SouthPolarStereo(central_longitude=0, true_scale_latitude=-71)

vectors = (
    ds[[vx, vy]]
    .sel(time="2019-01-01", method="nearest")
    .sel(x=slice(-1_800_000, -1_420_000), y=slice(-180_000, -760_000))
    .isel(x=slice(None, None, 80), y=slice(None, None, 80))
    .rename({vx: "vx", vy: "vy"})
    .copy()
)

vectors["speed"] = np.hypot(vectors.vx, vectors.vy)
vectors["angle"] = np.arctan2(vectors.vy, vectors.vx)

velocity_vectors = vectors.hvplot.vectorfield(
    x="x",
    y="y",
    angle="angle",
    mag="speed",
    color="#00e5ff",
    crs=proj3031,
    geo=True,
    tiles="EsriImagery",
    alpha=0.9,
    line_width=1.4,
    width=850,
    height=700,
    title=f"Ice velocity ({np.datetime_as_string(vectors.time.values, unit='D')})",
)

In [32]:
velocity_vectors

:Overlay
   .WMTS.I        :WMTS   [Longitude,Latitude]
   .VectorField.I :VectorField   [x,y]   (angle,speed)